# Fase 3.9 ? piloto TGIF/TGIF2

O TGIF oferece fotografias COCO e versoes alteradas vinculadas pelo mesmo ID.
Este notebook testa somente o **TGIF original + SD2 fully regenerated da
divisao validation do autor**. Baixa dois tar.gz completos (~1,76 GB) para
/content, confere hashes, nomes de arquivo, IDs compartilhados e algumas
imagens. A divisao validation da fonte serve apenas a este piloto; nao entra
no treino. Nao usa Mirage, OpenFake reservado, checkpoints ou GPU.

Depois do piloto, escolheremos apenas arquivos de **training** da fonte
para preparar e congelar um novo conjunto interno, com verificacao de
sobreposicao e divisoes do TCC. Nada e treinado ao abrir este notebook.


In [ ]:
# 1. Dependencias do piloto. GPU nao e necessaria.
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'cryptography', 'Pillow'])
print('Dependencias disponiveis.', flush=True)


In [ ]:
# 2. Monta o Drive; os pilotos leem somente o manifesto AIGI anterior.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
if not ROOT.is_dir():
    raise FileNotFoundError(ROOT)
print('Drive:', ROOT, '| livre:', round(shutil.disk_usage(ROOT).free / 1e9, 2), 'GB')
print('/content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')


In [ ]:
# 3. Desbloqueio seguro da credencial GitHub existente (formatos antigo e novo).
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

In [ ]:
# 4. Snapshot privado imutável da branch, somente leitura; sem token em URL ou Git.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'fase3-ajuste-fino'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase38-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('experimentos/fase3_9_tgif_piloto.py',)
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc37_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro os arquivo do piloto TGIF 3.9 na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

In [ ]:
# 5. Comandos e progresso periódico; nenhum treino é iniciado aqui.
import os, queue, subprocess, sys, threading, time
PILOT = CODE_ROOT / 'experimentos/fase3_9_tgif_piloto.py'
if not PILOT.is_file(): raise FileNotFoundError(PILOT)
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
def run_live(label, command, *, child_env=None):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, env=ENV if child_env is None else child_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (código {code}); veja o log acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('Código:', CODE_ROOT)

In [ ]:
# 6. PILOTO MANUAL TGIF: baixa originais + SD2-fr validation completos.
# Avisa durante downloads e verificacao. Nenhum dado de treino e criado.
run_live('piloto TGIF 3.9', [sys.executable, '-u', str(PILOT)])


In [ ]:
# 7. Resumo somente leitura. Envie esta saida para definir o preparo.
path = Path('/content/fase3_9_tgif_piloto/pilot_summary.json')
if not path.is_file():
    raise FileNotFoundError('Execute a celula 6 primeiro')
report = json.loads(path.read_text(encoding='utf-8'))
print(json.dumps(report, ensure_ascii=False, indent=2))
print('Nenhum treino, calibracao ou avaliacao externa foi iniciado.')
